# Inheritance in Python

Goal: understand why inheritance exists, how to use it, method overriding, `super()`, multi-level inheritance, and multiple inheritance with MRO.

## 1. What is Inheritance?

**Why we need it:** say you have a `Dog` class with a `name`, and methods like `eat()` and `sleep()`. Now you need a `Cat` class — it also needs a `name`, `eat()`, and `sleep()`, plus its own `meow()`. Copy-pasting `Dog`'s code into `Cat` works once, but every future fix to `eat()` now has to be made in two places, and that only gets worse as you add `Bird`, `Fish`, and so on. You want to write the shared parts once, in one place.

**Use case:** any time multiple classes share behavior/data but each also needs something of its own — different kinds of animals, different payment methods that all "process a payment," different UI widgets that all "render," different employee types that all get "paid."

**How to implement:** define a general class once (the **parent** / **base** / **superclass**), then define more specific classes that inherit from it (the **child** / **derived** / **subclass**) by putting the parent's name in parentheses. The child automatically gets every attribute and method the parent has, without rewriting them — it only needs to add what's different.

In [1]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        return f"{self.name} is eating"

    def sleep(self):
        return f"{self.name} is sleeping"


class Dog(Animal):       # Dog inherits from Animal
    def bark(self):
        return f"{self.name} says Woof!"


class Cat(Animal):       # Cat also inherits from Animal
    def meow(self):
        return f"{self.name} says Meow!"


rex = Dog("Rex")
tom = Cat("Tom")

print(rex.eat())      # inherited from Animal, not redefined in Dog
print(rex.bark())      # defined only in Dog
print(tom.sleep())    # inherited from Animal
print(tom.meow())      # defined only in Cat

print(isinstance(rex, Dog))     # True: rex is a Dog...
print(isinstance(rex, Animal))  # True: ...and every Dog is also an Animal
print(isinstance(tom, Dog))     # False: a Cat is not a Dog

Rex is eating
Rex says Woof!
Tom is sleeping
Tom says Meow!
True
True
False


## 2. Constructor Chaining

**Why we need it:** `Animal.__init__` is the code that actually sets up `self.name`. When `Dog` inherits from `Animal`, something needs to decide whether that setup code runs when you create a `Dog` — and if `Dog` adds its own `__init__` with extra setup of its own, both pieces of setup need to run, in the right order, without duplicating either one.

**Use case:** this comes up in literally every subclass that needs to store more data than its parent already does — you always need the parent's fields set up *and* the child's own fields set up.

**How to implement — the three cases:**
1. **Child has no `__init__` at all.** Python just uses the parent's `__init__` automatically. Nothing to chain — there's only one `__init__` in the picture.
2. **Child defines its own `__init__` and calls `super().__init__(...)`.** This is the normal, correct way to add fields: the parent's `__init__` runs first (setting up its part), then the rest of the child's `__init__` runs (setting up its own part). This is what "constructor chaining" means — each level runs its own setup by explicitly calling into the level above it.
3. **Child defines its own `__init__` but does NOT call `super().__init__(...)`.** The parent's `__init__` does **not** run on its own just because the class inherits from it — defining `__init__` in the child completely replaces the parent's, the same as overriding any other method. Whatever the parent's `__init__` would have set up (like `self.name`) simply never happens, unless the child sets it up itself. This is a common source of bugs.

In [1]:
class Animal:
    def __init__(self, name):
        self.name = name
        print(f"Animal.__init__ ran, set name={name!r}")


# Case 1: no __init__ at all -- Python just uses Animal's
class Dog(Animal):
    pass


print("--- Dog() ---")
rex = Dog("Rex")
print(rex.name)


# Case 2: defines __init__, chains via super().__init__()
class Cat(Animal):
    def __init__(self, name, color):
        super().__init__(name)          # chain: run Animal's setup first
        self.color = color
        print(f"Cat.__init__ ran, set color={color!r}")


print("--- Cat() ---")
tom = Cat("Tom", "black")
print(tom.name, tom.color)


# Case 3: defines __init__, forgets to call super().__init__() -- the bug
class BrokenBird(Animal):
    def __init__(self, name, can_fly):
        self.can_fly = can_fly          # only this line runs; Animal.__init__ never does
        print(f"BrokenBird.__init__ ran, set can_fly={can_fly!r}")


print("--- BrokenBird() ---")
tweety = BrokenBird("Tweety", True)
print(tweety.can_fly)
try:
    print(tweety.name)      # never got set, because Animal.__init__ never ran
except AttributeError as e:
    print(f"AttributeError: {e}")

--- Dog() ---
Animal.__init__ ran, set name='Rex'
Rex
--- Cat() ---
Animal.__init__ ran, set name='Tom'
Cat.__init__ ran, set color='black'
Tom black
--- BrokenBird() ---
BrokenBird.__init__ ran, set can_fly=True
True
AttributeError: 'BrokenBird' object has no attribute 'name'


## 3. Overriding Methods and `super()`

**Why we need it:** inheriting a method as-is is great when the behavior is identical, but often a child needs to do the *same thing plus a bit more*, or something *slightly different*. Copying the parent method into the child and editing it duplicates code again — the exact problem inheritance was meant to avoid.

**Use case:** a `Cat`'s `__init__` needs everything `Animal.__init__` already does (store `name`), plus something extra of its own (store `color`). Rewriting the `name`-storing line in `Cat` risks it drifting out of sync if `Animal.__init__` ever changes.

**How to implement:** define a method in the child class with the **same name** as one in the parent — this is called **overriding**; Python uses the child's version instead of the parent's. Inside the override, call `super().method(...)` to run the parent's original version first, then add whatever extra logic the child needs on top. (The `__init__` chaining from section 2 is really just this same rule, applied specifically to `__init__`.)

In [3]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"


class Cat(Animal):
    def __init__(self, name, color):
        super().__init__(name)   # runs Animal.__init__, which sets self.name
        self.color = color       # then add what's specific to Cat

    def speak(self):             # full override: completely different behavior, no super() call
        return f"{self.name} says Meow!"


class Cow(Animal):
    def speak(self):             # override that extends the parent instead of replacing it
        base_sound = super().speak()   # run Animal.speak() first
        return f"{base_sound}: Mooo!"  # then add to it


tom = Cat("Tom", "black")
print(tom.name, tom.color)
print(tom.speak())     # fully replaced

bessie = Cow("Bessie")
print(bessie.speak())  # extended, not replaced

Tom black
Tom says Meow!
Bessie makes a sound: Mooo!


## 4. Multi-level Inheritance

**Why we need it:** real hierarchies often have more than two levels. A `SportsCar` is a kind of `Car`, which is a kind of `Vehicle` — each level adding more specific behavior on top of the more general one below it.

**Use case:** modeling categories that naturally nest — `Vehicle` → `Car` → `SportsCar`, or `Employee` → `Manager` → `Director`.

**How to implement:** a class can inherit from a class that itself inherits from another class. Each level automatically has access to everything defined at every level above it, and `super()` inside a method always calls the *next* class up the chain (its immediate parent), not necessarily the very top one.

In [4]:
class Vehicle:
    def __init__(self, brand):
        self.brand = brand

    def describe(self):
        return f"A {self.brand} vehicle"


class Car(Vehicle):
    def __init__(self, brand, doors):
        super().__init__(brand)      # calls Vehicle.__init__
        self.doors = doors

    def describe(self):
        return f"{super().describe()} with {self.doors} doors"   # calls Vehicle.describe


class SportsCar(Car):
    def __init__(self, brand, doors, top_speed):
        super().__init__(brand, doors)   # calls Car.__init__ (not Vehicle directly)
        self.top_speed = top_speed

    def describe(self):
        return f"{super().describe()}, top speed {self.top_speed} km/h"   # calls Car.describe


sc = SportsCar("Ferrari", 2, 340)
print(sc.describe())          # built up from all three levels
print(sc.brand, sc.doors, sc.top_speed)

print(isinstance(sc, SportsCar), isinstance(sc, Car), isinstance(sc, Vehicle))

A Ferrari vehicle with 2 doors, top speed 340 km/h
Ferrari 2 340
True True True


## 5. Multiple Inheritance and MRO

**Why we need it:** sometimes something genuinely combines two unrelated capabilities that don't fit a single-parent chain — a `FlyingFish` that both `Swims` and `Flies`, where neither behavior is a specialization of the other.

**Use case:** mixing in independent, reusable pieces of behavior ("mixins") — e.g. a `LoggerMixin` that adds logging to any class, alongside whatever else that class already inherits.

**How to implement:** list more than one parent class in parentheses: `class FlyingFish(Swimmer, Flyer):`. Python then needs a rule for which parent to check first if both define the same method — that rule is the **MRO**, the **Method Resolution Order**. You can inspect it directly with `ClassName.__mro__` or `ClassName.mro()`. In practice: Python checks the class itself first, then its parents left-to-right in the order you listed them, going up the hierarchy — so the parent listed first "wins" if there's a name clash.

In [5]:
class Swimmer:
    def move(self):
        return "swimming"


class Flyer:
    def move(self):
        return "flying"


class FlyingFish(Swimmer, Flyer):   # Swimmer listed first
    pass


ff = FlyingFish()
print(ff.move())              # "swimming" -- Swimmer comes first in the parentheses

print([cls.__name__ for cls in FlyingFish.__mro__])   # the actual lookup order Python uses


class FlyingFish2(Flyer, Swimmer):  # swap the order
    pass


print(FlyingFish2().move())   # "flying" -- now Flyer comes first

swimming
['FlyingFish', 'Swimmer', 'Flyer', 'object']
flying


## 6. Putting It Together: an `Employee` hierarchy

**Why we need it:** to see single inheritance, constructor chaining, method overriding with `super()`, and a multi-level chain working together in one realistic example instead of isolated snippets.

**Use case:** a payroll system where every employee has a base salary calculation, but managers get a bonus on top, and senior managers get an additional bonus on top of that.

**How it's implemented here:** `Employee` defines the shared `__init__` and a base `pay()`. `Manager` chains into it with `super().__init__(...)` and overrides `pay()` to extend it with `super().pay()` plus a bonus. `SeniorManager` inherits from `Manager` (multi-level) and extends both `__init__` and `pay()` again the same way.

In [6]:
class Employee:
    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = base_salary

    def pay(self):
        return self.base_salary

    def __str__(self):
        return f"{self.name}: ${self.pay()}"


class Manager(Employee):
    def __init__(self, name, base_salary, bonus):
        super().__init__(name, base_salary)
        self.bonus = bonus

    def pay(self):
        return super().pay() + self.bonus


class SeniorManager(Manager):
    def __init__(self, name, base_salary, bonus, extra_bonus):
        super().__init__(name, base_salary, bonus)
        self.extra_bonus = extra_bonus

    def pay(self):
        return super().pay() + self.extra_bonus


staff = [
    Employee("Alice", 50000),
    Manager("Bob", 60000, bonus=5000),
    SeniorManager("Carol", 70000, bonus=8000, extra_bonus=3000),
]

for person in staff:
    print(person)

Alice: $50000
Bob: $65000
Carol: $81000
